# 02 Prepare FinanceBench Dataset

Σε αυτό το notebook προετοιμάζεται το αρχικό δείγμα του FinanceBench για χρήση στα επόμενα στάδια. Γίνεται φόρτωση των ερωτήσεων και των διαθέσιμων εγγράφων, έλεγχος βασικών πεδίων και δημιουργία του working dataset που χρησιμοποιείται στο parsing και στην αξιολόγηση.


In [ ]:
from pathlib import Path
import warnings

import pandas as pd


In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 140)

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PDFS_DIR = RAW_DIR / "pdfs"
INTERIM_DIR = DATA_DIR / "interim"

QUESTIONS_PATH = RAW_DIR / "financebench_open_source.jsonl"
DOC_INFO_PATH = RAW_DIR / "financebench_document_information.jsonl"

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PDFS_DIR:", PDFS_DIR)
print("QUESTIONS_PATH exists:", QUESTIONS_PATH.exists())
print("DOC_INFO_PATH exists:", DOC_INFO_PATH.exists())

In [ ]:
df_questions = pd.read_json(QUESTIONS_PATH, lines=True)
df_docs = pd.read_json(DOC_INFO_PATH, lines=True)

print("df_questions shape:", df_questions.shape)
print("df_docs shape:", df_docs.shape)

In [ ]:
print("Questions columns:")
for col in df_questions.columns:
    print("-", col)

print("\nDocument info columns:")
for col in df_docs.columns:
    print("-", col)

In [ ]:
display(df_questions.head(2))
display(df_docs.head(2))

In [ ]:
df_full = pd.merge(
    df_questions,
    df_docs,
    on="doc_name",
    how="left",
    suffixes=("", "_doc")
)

print("df_full shape:", df_full.shape)
df_full.head(3)

In [ ]:
merge_summary = {
    "question_rows": len(df_questions),
    "doc_rows": len(df_docs),
    "merged_rows": len(df_full),
    "missing_doc_metadata_rows": int(df_full["doc_type"].isna().sum()) if "doc_type" in df_full.columns else None,
    "unique_doc_names_questions": df_questions["doc_name"].nunique(),
    "unique_doc_names_docs": df_docs["doc_name"].nunique(),
}

pd.DataFrame([merge_summary])

In [ ]:
missing_df = pd.DataFrame({
    "column": df_full.columns,
    "missing_count": df_full.isna().sum().values,
    "missing_pct": (df_full.isna().mean().values * 100).round(2)
}).sort_values("missing_pct", ascending=False)

missing_df

In [ ]:
record = df_full.iloc[0].to_dict()

for k, v in record.items():
    print(f"{k}: {v}\n")

In [ ]:
print(type(df_full.loc[0, "evidence"]))
print(df_full.loc[0, "evidence"])

In [ ]:
first_evidence = df_full.loc[0, "evidence"][0] if len(df_full.loc[0, "evidence"]) > 0 else {}
first_evidence

In [ ]:
stats = {
    "n_questions": len(df_full),
    "n_unique_companies": df_full["company"].nunique(),
    "n_unique_docs": df_full["doc_name"].nunique(),
    "question_types": df_full["question_type"].nunique() if "question_type" in df_full.columns else None,
    "reasoning_types": df_full["question_reasoning"].nunique() if "question_reasoning" in df_full.columns else None,
}

pd.DataFrame([stats])

In [ ]:
if "question_type" in df_full.columns:
    display(df_full["question_type"].value_counts(dropna=False).to_frame("count"))

if "question_reasoning" in df_full.columns:
    display(df_full["question_reasoning"].value_counts(dropna=False).head(20).to_frame("count"))

if "doc_type" in df_full.columns:
    display(df_full["doc_type"].value_counts(dropna=False).to_frame("count"))

In [ ]:
pdf_files = sorted(PDFS_DIR.glob("*.pdf"))

pdf_inventory = pd.DataFrame({
    "pdf_filename": [p.name for p in pdf_files],
    "pdf_stem": [p.stem for p in pdf_files],
    "pdf_path": [str(p) for p in pdf_files]
})

print("Local PDFs found:", len(pdf_inventory))
pdf_inventory.head()

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

df_full["normalized_doc_name"] = df_full["doc_name"].apply(normalize_text)
pdf_inventory["normalized_pdf_stem"] = pdf_inventory["pdf_stem"].apply(normalize_text)

display(df_full[["doc_name", "normalized_doc_name"]].head())
display(pdf_inventory.head())

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = s.replace(".pdf", "")
    s = s.replace("_", " ")
    s = s.replace("-", " ")
    s = " ".join(s.split())
    return s

In [ ]:
df_matched = df_full.merge(
    pdf_inventory,
    left_on="normalized_doc_name",
    right_on="normalized_pdf_stem",
    how="left"
)

match_summary = {
    "total_rows": len(df_matched),
    "matched_rows": int(df_matched["pdf_filename"].notna().sum()),
    "unmatched_rows": int(df_matched["pdf_filename"].isna().sum()),
    "unique_docs_in_dataset": df_matched["doc_name"].nunique(),
    "unique_local_pdfs": len(pdf_inventory),
    "matched_unique_docs": df_matched.loc[df_matched["pdf_filename"].notna(), "doc_name"].nunique()
}

pd.DataFrame([match_summary])

In [ ]:
unmatched_docs = (
    df_matched.loc[df_matched["pdf_filename"].isna(), ["doc_name", "company", "doc_type", "doc_period"]]
    .drop_duplicates()
    .sort_values(["company", "doc_name"])
)

print("Unmatched unique docs:", len(unmatched_docs))
unmatched_docs.head(20)

In [ ]:
company_counts = (
    df_matched["company"]
    .value_counts()
    .reset_index()
)
company_counts.columns = ["company", "question_count"]

doc_counts = (
    df_matched["doc_name"]
    .value_counts()
    .reset_index()
)
doc_counts.columns = ["doc_name", "question_count"]

display(company_counts.head(15))
display(doc_counts.head(15))

In [ ]:
for col in ["question", "answer", "justification"]:
    if col in df_matched.columns:
        lengths = df_matched[col].fillna("").astype(str).str.len()
        print(f"\nColumn: {col}")
        print(lengths.describe())

In [ ]:
working_df = df_matched.copy().reset_index(drop=True)
working_df["row_id"] = working_df.index

priority_cols = [
    "row_id",
    "financebench_id",
    "question",
    "answer",
    "company",
    "doc_name",
    "doc_type",
    "doc_period",
    "pdf_filename",
    "pdf_path",
    "question_type",
    "question_reasoning",
    "justification",
    "evidence"
]

existing_priority_cols = [c for c in priority_cols if c in working_df.columns]
remaining_cols = [c for c in working_df.columns if c not in existing_priority_cols]

working_df = working_df[existing_priority_cols + remaining_cols]
working_df.head()

In [ ]:
working_csv_path = INTERIM_DIR / "financebench_open_source_working.csv"
working_parquet_path = INTERIM_DIR / "financebench_open_source_working.parquet"

working_df.to_csv(working_csv_path, index=False)
print("Saved CSV to:", working_csv_path)

try:
    working_df.to_parquet(working_parquet_path, index=False)
    print("Saved Parquet to:", working_parquet_path)
except ImportError as e:
    print("Parquet save skipped.")
    print("Reason:", e)
    print("Install pyarrow with: pip install pyarrow")

In [ ]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

In [ ]:
output_path = DATA_DIR / "interim" / "financebench_sample_working.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

working_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

In [ ]:
unmatched_path = INTERIM_DIR / "financebench_unmatched_docs.csv"
unmatched_docs.to_csv(unmatched_path, index=False)

print("Saved unmatched docs to:", unmatched_path)

## Συμπέρασμα

Σε αυτό το notebook:

- φορτώσαμε τα δύο JSONL αρχεία του FinanceBench sample
- ενώσαμε questions και document metadata με βάση το `doc_name`
- ελέγξαμε τα τοπικά PDFs
- κάναμε πρώτη αντιστοίχιση dataset documents ↔ local files
- αποθηκεύσαμε working dataset για τα επόμενα στάδια

Το επόμενο notebook είναι το `03_parse_pdfs_with_docling.ipynb`.